# Lab 5 — Performance Optimization

**ROCm Certification Program — Level 1**

<div style="
    background: linear-gradient(135deg, #eef6ff 0%, #e8f5ee 100%);
    border-left: 6px solid #4f8ad9;
    padding: 18px;
    border-radius: 10px;
    margin: 20px 0;
">
<h2 style="margin-top:0; color:#1f4f99;">🎯 Objectives</h2>

<p>In this lab, you will explore several common GPU optimization techniques and observe how they affect performance.</p>

<ul>
<li>Improve matrix-transpose performance using LDS (shared memory)</li>
<li>Measure the effect of workgroup size on memory-bound kernels</li>
<li>Explore mixed-precision computation using FP32, FP16, and BF16</li>
<li>Learn that GPU optimization is an experimental process: measure → optimize → validate</li>
</ul>
</div>


## Part 1 — Baseline Measurement

<div style="
    background:#eef6ff;
    border-left:6px solid #4f8ad9;
    padding:16px;
    border-radius:8px;
    margin:15px 0;
">

<h3 style="margin-top:0; color:#2c5282;">
🔍 Establishing a Performance Baseline
</h3>

<p>
Before optimizing a GPU kernel, we first need to understand its current
performance characteristics.
</p>

<p>
This initial measurement is called the <b>baseline</b>.
All future optimizations will be compared against it.
</p>

</div>

### Why Matrix Transpose?

A matrix transpose swaps rows and columns:

```text
Original Matrix          Transposed Matrix

1  2  3                  1  4  7
4  5  6        --->      2  5  8
7  8  9                  3  6  9
```

Although the operation is mathematically simple, it is widely used in:

- Scientific computing
- Linear algebra libraries
- Machine learning workloads
- Data layout transformations

More importantly, matrix transpose is an excellent demonstration of how
memory access patterns affect GPU performance.

### The Deliberate Bottleneck

In this exercise, the transpose kernel contains a deliberate inefficiency.

Reading from memory occurs sequentially, which is generally efficient.

However, the output elements are written using a **strided** pattern:

```text
Read:   contiguous addresses  ✓

Write:  scattered addresses   ✗
```

This results in **uncoalesced memory accesses**.

<div style="
    background:#fff8e8;
    border-left:6px solid #d8c27a;
    padding:14px;
    border-radius:8px;
    margin:15px 0;
">

<b>What is Memory Coalescing?</b><br><br>

GPUs achieve high memory bandwidth when neighboring threads access
neighboring memory locations.

The hardware can then combine many memory requests into a small number
of large transactions.

When threads access memory with large strides or irregular patterns,
more memory transactions are required and effective bandwidth decreases.

</div>

### Goal of This Exercise

We will:

1. Measure the performance of the baseline kernel
2. Use profiling tools to identify the bottleneck
3. Observe the impact of uncoalesced memory accesses
4. Establish a reference point for later optimizations

<div style="
    background:#eef8ee;
    border-left:5px solid #68a36d;
    padding:12px;
    border-radius:8px;
    margin:15px 0;
">

<b>Key Question</b><br><br>

Is the kernel limited by computation or by memory access efficiency?

The profiler will help us answer that question.

</div>

<div style="
background:#fff8e8;
border-left:6px solid #d8c27a;
padding:14px;
border-radius:8px;
margin:15px 0;
">

<b>Prediction Exercise</b><br><br>

Before running the benchmark, try to predict:

<ul>
<li>Which kernel will be faster?</li>
<li>Why?</li>
<li>Will the optimized version perform more arithmetic operations?</li>
</ul>

The answer illustrates an important GPU principle:

<b>Better memory access patterns often improve performance more than additional computation.</b>

</div>

In [ ]:
%%writefile transpose.cpp
#include <hip/hip_runtime.h>
#include <cstdio>
#include <cstdlib>

//
// BASELINE VERSION
//
// Simple matrix transpose.
//
// Reads are contiguous and therefore reasonably efficient.
//
// Writes, however, are strided:
//
//     out[x * H + y]
//
// Neighboring threads write to memory locations that are
// far apart, producing uncoalesced memory accesses.
//
__global__ void transpose_naive(
    const float* in,
    float* out,
    int W,
    int H)
{
    int x =
        blockIdx.x * blockDim.x + threadIdx.x;

    int y =
        blockIdx.y * blockDim.y + threadIdx.y;

    if (x < W && y < H)
    {
        //
        // Read:
        //   contiguous
        //
        // Write:
        //   strided / uncoalesced
        //
        out[x * H + y] =
            in[y * W + x];
    }
}

//
// OPTIMIZED VERSION
//
// Uses LDS (Local Data Share / shared memory)
// as a temporary staging buffer.
//
// Threads first load a tile of the matrix
// into shared memory, then write it back
// in transposed order.
//
// This converts the global-memory accesses
// into largely coalesced reads and writes.
//
#define TILE 16

__global__ void transpose_tiled(
    const float* in,
    float* out,
    int W,
    int H)
{
    //
    // Shared-memory tile.
    //
    // The extra column (+1) avoids
    // shared-memory bank conflicts when
    // threads access the tile transposed.
    //
    __shared__ float tile[TILE][TILE + 1];

    //
    // Coordinates in the input matrix.
    //
    int xIn =
        blockIdx.x * TILE + threadIdx.x;

    int yIn =
        blockIdx.y * TILE + threadIdx.y;

    //
    // Load a tile from global memory
    // into shared memory.
    //
    if (xIn < W && yIn < H)
    {
        tile[threadIdx.y][threadIdx.x] =
            in[yIn * W + xIn];
    }

    //
    // Wait until the entire tile has
    // been loaded.
    //
    __syncthreads();

    //
    // Coordinates in the output matrix.
    //
    int xOut =
        blockIdx.y * TILE + threadIdx.x;

    int yOut =
        blockIdx.x * TILE + threadIdx.y;

    //
    // Store the transposed tile.
    //
    // Accessing the tile through
    // [x][y] performs the transpose.
    //
    if (xOut < H && yOut < W)
    {
        out[yOut * H + xOut] =
            tile[threadIdx.x][threadIdx.y];
    }
}

//
// Benchmark helper.
//
// Runs the kernel multiple times and
// reports average execution time and
// effective memory bandwidth.
//
void run_kernel(
    const char* name,
    void(*kernel)(const float*,float*,int,int),
    float* d_in,
    float* d_out,
    int W,
    int H)
{
    dim3 block(TILE, TILE);

    dim3 grid(
        (W + TILE - 1) / TILE,
        (H + TILE - 1) / TILE);

    //
    // Warm-up launch.
    //
    // Removes one-time startup effects
    // from timing measurements.
    //
    kernel<<<grid, block>>>(
        d_in, d_out, W, H);

    hipDeviceSynchronize();

    hipEvent_t t0, t1;

    hipEventCreate(&t0);
    hipEventCreate(&t1);

    //
    // Timed runs.
    //
    hipEventRecord(t0);

    for (int i = 0; i < 50; i++)
    {
        kernel<<<grid, block>>>(
            d_in, d_out, W, H);
    }

    hipEventRecord(t1);
    hipEventSynchronize(t1);

    float ms;

    hipEventElapsedTime(
        &ms,
        t0,
        t1);

    ms /= 50;

    //
    // Effective bandwidth:
    //
    // Read matrix + write matrix.
    //
    double bw =
        2.0 * W * H * sizeof(float)
        / (ms * 1e-3)
        / 1e9;

    printf(
        "%-20s Time: %7.3f ms   Bandwidth: %6.1f GB/s\n",
        name,
        ms,
        bw);

    hipEventDestroy(t0);
    hipEventDestroy(t1);
}

int main()
{
    //
    // Matrix dimensions.
    //
    int W = 4096;
    int H = 4096;
    //int W = 16384;
    //int H = 16384;

    size_t bytes =
        W * H * sizeof(float);

    //
    // Host memory.
    //
    float* h_in =
        (float*)malloc(bytes);

    //
    // Initialize input matrix.
    //
    for (int i = 0; i < W * H; i++)
    {
        h_in[i] = (float)i;
    }

    //
    // Device memory.
    //
    float *d_in, *d_out;

    hipMalloc(&d_in, bytes);
    hipMalloc(&d_out, bytes);

    hipMemcpy(
        d_in,
        h_in,
        bytes,
        hipMemcpyHostToDevice);

    printf(
        "Matrix Transpose %dx%d\n\n",
        W,
        H);

    //
    // Compare naive and optimized versions.
    //
    run_kernel(
        "Naive (uncoalesced)",
        transpose_naive,
        d_in,
        d_out,
        W,
        H);

    run_kernel(
        "Tiled (LDS)",
        transpose_tiled,
        d_in,
        d_out,
        W,
        H);

    hipFree(d_in);
    hipFree(d_out);

    free(h_in);

    return 0;
}

In [ ]:
import subprocess
subprocess.run(['hipcc','-O3','-o','transpose','transpose.cpp'],
               capture_output=True, text=True, check=True)
result = subprocess.run(['./transpose'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Run the transpose benchmark again after compilation.
#
# Earlier labs introduced rocprofv3 and GPU profiling.
# In this lab we focus on performance measurements and
# optimization techniques rather than profiler output.

import subprocess

result = subprocess.run(
    ['./transpose'],
    capture_output=True,
    text=True
)

print(result.stdout)


<div style="
    background:#fff8e8;
    border-left:6px solid #d8c27a;
    padding:14px;
    border-radius:8px;
    margin:15px 0;
">

<b>Experiment: Does Size Matter?</b><br><br>

The default matrix size is:

<pre>
4096 × 4096
</pre>

This matrix occupies approximately:

<pre>
64 MB
</pre>

on the GPU and may still benefit from cache effects.

As a result, the performance difference between the naive and tiled
implementations can appear relatively small.

<br>

<b>Try the following experiment:</b>

Locate these lines:

<pre>
int W = 4096;
int H = 4096;
</pre>

and change them to:

<pre>
int W = 16384;
int H = 16384;
</pre>

The larger matrix occupies approximately:

<pre>
1 GB
</pre>

and is much less likely to fit in cache.

<br>

<b>Questions:</b>

<ul>
<li>How does the bandwidth of the naive version change?</li>
<li>How does the bandwidth of the tiled version change?</li>
<li>Does the performance gap become larger or smaller?</li>
<li>What does this tell you about the role of memory-access patterns?</li>
</ul>

<b>Expected observation:</b>

For larger matrices, the tiled implementation typically shows a
significantly greater advantage because coalesced memory accesses
become more important once cache effects diminish.

</div>

## Part 2 — Workgroup Size Sweep

<div style="
    background:#eef6ff;
    border-left:6px solid #4f8ad9;
    padding:16px;
    border-radius:8px;
    margin:15px 0;
">

<h3 style="margin-top:0; color:#2c5282;">
⚙️ Finding the Right Workgroup Size
</h3>

<p>
GPU performance depends not only on the kernel itself, but also on how
threads are organized into workgroups (thread blocks).
</p>

<p>
In this exercise, we will benchmark several workgroup sizes and observe
their effect on execution time and throughput.
</p>

</div>

### What Is a Workgroup?

A workgroup is a collection of threads that execute together on a Compute Unit (CU).

Examples:

```text
64 threads   = 1 wavefront
128 threads  = 2 wavefronts
256 threads  = 4 wavefronts
512 threads  = 8 wavefronts
```

Each workgroup shares:

- Local Data Share (LDS / shared memory)
- Synchronization primitives
- Compute Unit resources

Choosing the workgroup size determines how efficiently these resources
are utilized.

---

### Why Does Workgroup Size Matter?

A common assumption is:

```text
More threads = More performance
```

In practice, the answer is more complicated.

Larger workgroups can:

✅ Increase occupancy

✅ Hide memory latency

✅ Keep more execution units busy

But they can also:

❌ Consume more registers

❌ Use more LDS

❌ Reduce the number of resident workgroups per CU

As a result, the largest workgroup size is not always the fastest.

---

### Occupancy Refresher

Occupancy describes how many wavefronts can be active on a Compute Unit
at the same time.

Higher occupancy often helps hide:

- Memory latency
- Cache misses
- Pipeline stalls

However, after a certain point, increasing occupancy may provide little
or no additional benefit.

<div style="
    background:#fff8e8;
    border-left:6px solid #d8c27a;
    padding:14px;
    border-radius:8px;
    margin:15px 0;
">

<b>Important</b><br><br>

High occupancy does <b>not</b> automatically mean high performance.

The goal is not to maximize occupancy at all costs.

The goal is to maximize throughput.

Sometimes a kernel with lower occupancy can outperform one with higher
occupancy because it uses resources more efficiently.

</div>

### Goal of This Exercise

We will:

1. Run the same kernel using multiple workgroup sizes
2. Measure execution time and throughput
3. Observe how occupancy affects performance
4. Identify the "sweet spot" for this particular kernel

<div style="
    background:#eef8ee;
    border-left:5px solid #68a36d;
    padding:12px;
    border-radius:8px;
    margin:15px 0;
">

<b>Engineering Question</b><br><br>

If larger workgroups expose more parallelism, why don't we always use
the largest possible workgroup size?

The measurements in this section will help answer that question.

</div>

---

In [ ]:
%%writefile sweep.cpp
#include <hip/hip_runtime.h>
#include <cstdio>

//
// SAXPY = Single-Precision A*X Plus Y
//
// One of the classic BLAS operations:
//
//     y = a * x + y
//
// The kernel performs very little computation
// per element and is therefore typically
// memory-bandwidth limited.
//
__global__ void saxpy(
    float a,
    const float* x,
    float* y,
    int N)
{
    int i =
        blockDim.x * blockIdx.x +
        threadIdx.x;

    if (i < N)
    {
        y[i] =
            a * x[i] + y[i];
    }
}

int main()
{
    //
    // Number of elements.
    //
    // 2^24 = 16,777,216 floats
    //
    int N = 1 << 24;

    size_t bytes =
        N * sizeof(float);

    //
    // Device memory.
    //
    float *d_x, *d_y;

    hipMalloc(&d_x, bytes);
    hipMalloc(&d_y, bytes);

    //
    // Initialize vectors.
    //
    // Exact values are unimportant;
    // we only care about performance.
    //
    hipMemset(d_x, 1, bytes);
    hipMemset(d_y, 1, bytes);

    //
    // Candidate block sizes.
    //
    // 64 threads  = 1 wavefront
    // 128 threads = 2 wavefronts
    // 256 threads = 4 wavefronts
    // 512 threads = 8 wavefronts
    // 1024 threads = 16 wavefronts
    //
    int sizes[] =
    {
        64,
        128,
        256,
        512,
        1024
    };

    printf(
        "%-15s %-12s %-12s\n",
        "Block Size",
        "Time (ms)",
        "BW (GB/s)");

    printf(
        "%-15s %-12s %-12s\n",
        "----------",
        "---------",
        "---------");

    //
    // Sweep through several
    // workgroup sizes and compare
    // performance.
    //
    for (int b = 0; b < 5; b++)
    {
        int bs =
            sizes[b];

        int grid =
            (N + bs - 1) / bs;

        //
        // Warm-up launch.
        //
        saxpy<<<grid, bs>>>(
            2.0f,
            d_x,
            d_y,
            N);

        hipDeviceSynchronize();

        hipEvent_t t0, t1;

        hipEventCreate(&t0);
        hipEventCreate(&t1);

        //
        // Timed launches.
        //
        hipEventRecord(t0);

        for (int i = 0; i < 50; i++)
        {
            saxpy<<<grid, bs>>>(
                2.0f,
                d_x,
                d_y,
                N);
        }

        hipEventRecord(t1);
        hipEventSynchronize(t1);

        float ms;

        hipEventElapsedTime(
            &ms,
            t0,
            t1);

        ms /= 50;

        //
        // Effective bandwidth.
        //
        // For each element:
        //
        //   Read x[i]
        //   Read y[i]
        //   Write y[i]
        //
        // Total traffic:
        //
        //   3 × N × sizeof(float)
        //
        double bw =
            3.0 * N * sizeof(float)
            / (ms * 1e-3)
            / 1e9;

        printf(
            "%-15d %-12.3f %-12.1f\n",
            bs,
            ms,
            bw);

        hipEventDestroy(t0);
        hipEventDestroy(t1);
    }

    hipFree(d_x);
    hipFree(d_y);

    return 0;
}

In [ ]:
import subprocess
subprocess.run(['hipcc','-O3','-o','sweep','sweep.cpp'],
               capture_output=True, text=True, check=True)
result = subprocess.run(['./sweep'], capture_output=True, text=True)
print(result.stdout)


<div style="
    background:#eef8ee;
    border-left:5px solid #68a36d;
    padding:12px;
    border-radius:8px;
    margin:15px 0;
">

<b>Results Analysis</b><br><br>

Modern GPUs are remarkably good at hiding latency by executing many wavefronts concurrently. As a result, performance often improves rapidly as the workgroup size increases and then reaches a plateau.

Different GPU architectures may favor different workgroup sizes. In many cases there is no single "magic number" that is always optimal.

The important observation is that GPU performance tuning is empirical:

<ul>
<li>Measure performance.</li>
<li>Experiment with launch parameters.</li>
<li>Choose the configuration that performs best for your workload.</li>
</ul>

For memory-bound kernels such as SAXPY, several workgroup sizes may deliver very similar performance once the GPU reaches sufficient occupancy.

</div>


<div style="
    background:#fff8e8;
    border-left:6px solid #d8c27a;
    padding:14px;
    border-radius:8px;
    margin:15px 0;
">

<b>Engineering Lesson</b><br><br>

Many developers assume that increasing the workgroup size will always
improve performance.

This experiment demonstrates why GPU optimization should be guided by
measurement rather than intuition.

The optimal workgroup size depends on:

<ul>
<li>GPU architecture</li>
<li>Register usage</li>
<li>LDS usage</li>
<li>Memory-access behavior</li>
<li>Kernel complexity</li>
</ul>

The fastest configuration is often found experimentally.

</div>

---
## Part 3 — Mixed Precision: FP32 vs FP16

<div style="
    background:#eef6ff;
    border-left:6px solid #4f8ad9;
    padding:16px;
    border-radius:8px;
    margin:15px 0;
">

<h3 style="margin-top:0; color:#2c5282;">
⚡ Trading Precision for Performance
</h3>

<p>
Modern GPUs can often process lower-precision data significantly faster
than standard 32-bit floating-point values.
</p>

<p>
This technique is known as <b>mixed-precision computing</b> and is widely
used in machine learning, scientific computing, and high-performance
applications.
</p>

</div>

### Why Use FP16?

Most GPU applications do not require every value to be represented with
full FP32 precision.

By storing values as 16-bit floating-point numbers (FP16):

- Memory usage is reduced by 50%
- Memory bandwidth requirements are reduced
- More values fit into caches
- Specialized hardware units can process more operations per cycle

As a result, many GPUs can execute FP16 workloads substantially faster
than equivalent FP32 workloads.

---

### FP32 vs FP16

| Format | Size | Typical Precision |
|----------|----------|----------|
| FP32 | 32 bits | ~7 decimal digits |
| FP16 | 16 bits | ~3–4 decimal digits |

FP16 uses half as much storage but also provides less numerical precision.

The key engineering question becomes:

> **Is the loss of precision acceptable for the application?**

---

### Mixed Precision in Practice

Today, mixed precision is widely used in:

- Deep learning training
- Deep learning inference
- Computer vision
- Signal processing
- Large-scale HPC workloads

Many modern AI models perform most computations in FP16 or BF16 while
retaining critical accumulations in higher precision.

---

### Goal of This Exercise

We will:

1. Run the same computation using FP32
2. Run the same computation using FP16
3. Measure performance differences
4. Compare numerical results
5. Determine whether the loss of precision is acceptable

<div style="
    background:#fff8e8;
    border-left:6px solid #d8c27a;
    padding:14px;
    border-radius:8px;
    margin:15px 0;
">

<b>Important</b><br><br>

Faster is not always better.

Mixed-precision optimization is only useful if the resulting numerical
error remains acceptable for the application.

Every performance gain must be balanced against accuracy requirements.

</div>

---

### What Should You Expect?

On modern AMD Instinct GPUs, FP16 operations can achieve significantly
higher throughput than FP32.

However, the exact speedup depends on:

<ul>
<li>GPU architecture</li>
<li>Arithmetic intensity</li>
<li>Memory bandwidth</li>
<li>Compiler optimizations</li>
<li>The specific kernel being executed</li>
</ul>

The purpose of this exercise is not simply to measure a speedup, but to
understand the tradeoff between performance and numerical accuracy.

<div style="
    background:#eef8ee;
    border-left:5px solid #68a36d;
    padding:12px;
    border-radius:8px;
    margin:15px 0;
">

<b>Engineering Question</b><br><br>

If FP16 is faster and uses less memory, why don't all applications use it?

The answer lies in the balance between performance and numerical accuracy.

</div>

---

In [ ]:
# Mixed-precision GEMM benchmark using PyTorch
#
# PyTorch automatically uses rocBLAS on ROCm for matrix multiplication.
#
# We compare three floating-point formats:
#
#   FP32  - standard single precision
#   FP16  - half precision
#   BF16  - Brain Floating Point
#
# The goal is to observe the tradeoff between:
#
#   • Performance
#   • Numerical precision
#
import torch
import time

device = torch.device('cuda')

#
# Matrix dimensions.
#
# GEMM performs:
#
#   C = A × B
#
# with:
#
#   A = M × K
#   B = K × N
#   C = M × N
#
M, N, K = 2048, 2048, 2048


def bench_gemm(dtype, label):

    #
    # Create random input matrices
    # in the selected precision.
    #
    A = torch.randn(
        M, K,
        dtype=dtype,
        device=device)

    B = torch.randn(
        K, N,
        dtype=dtype,
        device=device)

    #
    # Warm-up run.
    #
    # Excludes one-time initialization
    # and autotuning overhead from
    # timing measurements.
    #
    torch.mm(A, B)
    torch.cuda.synchronize()

    #
    # Timed runs.
    #
    torch.cuda.synchronize()

    t0 = time.perf_counter()

    for _ in range(50):
        C = torch.mm(A, B)

    torch.cuda.synchronize()

    ms = (
        time.perf_counter() - t0
    ) / 50 * 1000

    #
    # GEMM requires:
    #
    #   2 × M × N × K FLOPs
    #
    gflops = (
        2 * M * N * K
        / (ms * 1e-3)
        / 1e9
    )

    print(
        f"{label:10s}  "
        f"Time: {ms:7.3f} ms   "
        f"GFLOPS: {gflops:8.1f}"
    )

    return C


print(f"GEMM {M}x{N}x{K}\n")

#
# Baseline.
#
c32 = bench_gemm(
    torch.float32,
    "FP32")

#
# Half precision.
#
c16 = bench_gemm(
    torch.float16,
    "FP16")

#
# Brain Floating Point.
#
# BF16 keeps the same exponent
# range as FP32 but uses fewer
# mantissa bits.
#
# This often provides better
# numerical stability than FP16
# while still benefiting from
# reduced precision hardware.
#
cbf = bench_gemm(
    torch.bfloat16,
    "BF16")


<div style="
    background:#eef6ff;
    border-left:6px solid #4f8ad9;
    padding:14px;
    border-radius:8px;
    margin:15px 0;
">

<b>Exercise</b><br><br>

Using your measured execution times, calculate:

<pre>
FP16 Speedup = FP32 Time ÷ FP16 Time

BF16 Speedup = FP32 Time ÷ BF16 Time
</pre>

Compare the results to the published peak-performance ratios for your GPU.

</div>

<div style="
background:#fff8e8;
border-left:6px solid #d8c27a;
padding:14px;
border-radius:8px;
margin:15px 0;
">

<b>Important Caveat</b><br><br>

This comparison is not entirely "apples to apples."

FP16 and BF16 achieve higher throughput because they process
lower-precision values than FP32.

The observed speedup does not mean the GPU suddenly became
4× faster at the same computation. Instead, it is performing
a different numerical computation with reduced precision.

Whether this tradeoff is acceptable depends on the application.

</div>

<div style="
background:#eef8ee;
border-left:5px solid #68a36d;
padding:12px;
border-radius:8px;
margin:15px 0;
">

<b>Key Takeaway</b><br><br>

Mixed precision is not a free optimization.

It trades numerical accuracy for higher performance,
lower memory usage, and greater hardware utilization.

The engineer's job is to determine whether the resulting
accuracy remains acceptable for the target application.

</div>

---

## Part 5 — Results Summary

Fill in the table below using your measured results.

| Optimization | Time (ms) | Bandwidth/GFLOPS | Improvement |
|-------------|-----------|------------------|-------------|
| Transpose naive | | | baseline |
| Transpose tiled (LDS) | | | |
| Best workgroup size | | | |
| FP32 GEMM | | | |
| FP16 GEMM | | | |
| BF16 GEMM | | | |

---

## Congratulations!

You have completed all Level 1 labs. You are ready for the certification exam.

### Skills demonstrated:
- ✅ ROCm environment setup via Docker
- ✅ GPU architecture understanding (memory hierarchy, roofline)
- ✅ Iterative kernel tuning guided by profiler data
- ✅ ROCm library usage (rocBLAS, MIOpen, rocFFT)
- ✅ PyTorch training and debugging on ROCm
- ✅ HIP kernel writing and CUDA porting
- ✅ Performance optimization (coalescing, workgroup tuning, mixed precision)